In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib

In [ ]:
data = {

    "opportunity_id": [
        1, 2, 3, 4, 5
    ],

    "title": [
        "AI Internship",
        "Web Development Training",
        "Data Science Program",
        "Mobile App Development",
        "Cyber Security Training"
    ],


    "skills": [
        "python machine learning deep learning tensorflow",
        "html css javascript react frontend",
        "python sql data analysis machine learning",
        "flutter dart mobile development",
        "network security linux cybersecurity"
    ]
}


df = pd.DataFrame(data)


df.head()

In [ ]:
vectorizer = TfidfVectorizer()

skill_vectors = vectorizer.fit_transform(
    df["skills"]
)
print("Recommendation Model trained successfully")

In [ ]:
def recommend_opportunities(student_skills, top_n=3):
    student_vector = vectorizer.transform(
        [student_skills]
    )

    similarity_scores = cosine_similarity(
        student_vector,
        skill_vectors
    )[0]


    top_indices = similarity_scores.argsort(
        )[::-1][:top_n]


    recommendations = []


    for index in top_indices:

        recommendations.append({

            "title":
                df.iloc[index]["title"],

            "match_score":
                round(
                    float(similarity_scores[index]),
                    2
                )
        })


    return recommendations


joblib.dump(
    vectorizer,
    "recommendation_vectorizer.pkl"
)


joblib.dump(
    skill_vectors,
    "opportunity_vectors.pkl"
)


joblib.dump(
    df,
    "opportunities_data.pkl"
)


print("Recommendation system saved successfully")

In [ ]:
student_skills = (
    "python machine learning sql"
)
recommendations = recommend_opportunities(
    student_skills
)

recommendations

In [ ]:
from fastapi import FastAPI
import joblib
from sklearn.metrics.pairwise import cosine_similarity


app = FastAPI(
    title="Personalized Training Recommendations AI API",
    description="Recommend best training opportunities using AI",
    version="1.0"
)

vectorizer = joblib.load(
    "recommendation_vectorizer.pkl"
)


skill_vectors = joblib.load(
    "opportunity_vectors.pkl"
)

opportunities = joblib.load(
    "opportunities_data.pkl"
)


print("Recommendation AI Model loaded successfully")

In [ ]:
@app.get("/")
def home():

    return {
        "message": "Hello World"
    }

@app.post("/training-recommendations")
def training_recommendations(data: dict):

    student_id = data.get(
        "studentId"
    )


    student_skills = data.get(
        "student_skills",
        []
    )

    skills_text = " ".join(
        [
            skill.lower()
            for skill in student_skills
        ]
    )

    student_vector = vectorizer.transform(
        [skills_text]
    )

    similarity_scores = cosine_similarity(
        student_vector,
        skill_vectors
    )[0]

    top_indices = similarity_scores.argsort(
    )[::-1][:3]


    recommendations = []


    for index in top_indices:

        recommendations.append({

            "title":
                opportunities.iloc[index]["title"],

            "match_score":
                round(
                    float(similarity_scores[index]),
                    2
                )
        })


    return {

        "studentId":
            student_id,


        "recommended_opportunities":
            recommendations
    }

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()
config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8001,
    log_level="info"
)

server = uvicorn.Server(config)
await server.serve()